In [2]:
import sys
print("Python:", sys.executable)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_colwidth", 150)

df = pd.read_csv("../data/processed/complaints_sample.csv")

print(f"\nShape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

Python: c:\PROJECTS\3-FINTECH PAYMENT TRIAGE PROJECT\venv\Scripts\python.exe

Shape: (161452, 5)
Columns: ['Product', 'Sub-product', 'Issue', 'Consumer complaint narrative', 'Complaint ID']


,Product,Sub-product,Issue,Consumer complaint narrative,Complaint ID
0,Bank account or service,Checking account,Problems caused by my funds being low,I was charged on a XXXX NSF fee because my account was overdrawn {$0.00} cent due the Bank of America moving the {$0.00} to the other account from...,1852816
1,Bank account or service,Checking account,"Account opening, closing, or management",I have an overdraft account with TD Bank. The statements come the first week of the month with a due date the last day of the month. I will pay th...,1929157
2,Bank account or service,Checking account,Problems caused by my funds being low,Between XXXX and XXXX I was charged excessive overdraft fees. Some of those fees were transfers to my own Bank of America checking or savings acco...,2141289


In [3]:
print(f"Unique Products: {df['Product'].nunique()}\n")

product_counts = df["Product"].value_counts()
print(product_counts)

Unique Products: 21

Product
Bank account or service                                                         9523
Checking or savings account                                                     9523
Credit card                                                                     9523
Credit card or prepaid card                                                     9523
Credit reporting                                                                9523
Credit reporting or other personal consumer reports                             9523
Credit reporting, credit repair services, or other personal consumer reports    9523
Debt collection                                                                 9523
Money transfer, virtual currency, or money service                              9523
Mortgage                                                                        9523
Payday loan, title loan, or personal loan                                       9523
Payday loan, title loan, personal lo

In [4]:
sample_product = "Credit card"
subset = df[df["Product"] == sample_product]

print(f"Product: {sample_product} — {len(subset)} rows")
print(f"Unique Issues: {subset['Issue'].nunique()}\n")
print(subset["Issue"].value_counts().head(20))

Product: Credit card — 9523 rows
Unique Issues: 43

Issue
Problem with a purchase shown on your statement                    2159
Fees or interest                                                    966
Other features, terms, or problems                                  945
Getting a credit card                                               930
Problem with a company's investigation into an existing problem     648
Closing your account                                                632
Incorrect information on your report                                513
Problem when making payments                                        492
Advertising and marketing, including promotional offers             391
Trouble using your card                                             324
Billing disputes                                                    219
Identity theft / Fraud / Embezzlement                               138
Struggling to pay your bill                                         105
Other 

In [5]:
sample_product = "Money transfer, virtual currency, or money service"
subset = df[df["Product"] == sample_product]

print(f"Product: {sample_product} — {len(subset)} rows")
print(f"Unique Issues: {subset['Issue'].nunique()}\n")
print(subset["Issue"].value_counts().head(20))

Product: Money transfer, virtual currency, or money service — 9523 rows
Unique Issues: 17

Issue
Other transaction problem                                   4328
Fraud or scam                                               2068
Unauthorized transactions or other transaction problem       758
Money was not available when promised                        529
Managing, opening, or closing your mobile wallet account     470
Trouble accessing funds in your mobile or digital wallet     318
Other service problem                                        270
Confusing or missing disclosures                             245
Unexpected or other fees                                     135
Problem with customer service                                122
Wrong amount charged or received                              97
Problem adding money                                          59
Confusing or misleading advertising or marketing              53
Lost or stolen money order                                

In [6]:
import sys
sys.path.insert(0, "../src")

from features.label_mapping import map_issue_to_label, KEEP_PRODUCTS

# Test the mapping on a few examples
test_issues = [
    "Fraud or scam",
    "Problem with a purchase shown on your statement",
    "Problem when making payments",
    "Getting a credit card",
    "Incorrect information on your report",
    "Identity theft / Fraud / Embezzlement",
    "Other transaction problem",
    "Fees or interest",
]

for issue in test_issues:
    label = map_issue_to_label(issue)
    print(f"{str(label):20s} <- {issue}")

fraud                <- Fraud or scam
billing_dispute      <- Problem with a purchase shown on your statement
payment_issue        <- Problem when making payments
account_issue        <- Getting a credit card
None                 <- Incorrect information on your report
fraud                <- Identity theft / Fraud / Embezzlement
payment_issue        <- Other transaction problem
billing_dispute      <- Fees or interest


In [7]:
from features.label_mapping import KEEP_PRODUCTS, map_issue_to_label

# Filter to only the products we care about
df_kept = df[df["Product"].isin(KEEP_PRODUCTS)].copy()
print(f"Rows after product filter: {len(df_kept):,}")

# Apply the label mapping
df_kept["category"] = df_kept["Issue"].apply(map_issue_to_label)

# How many did we successfully label?
labeled = df_kept[df_kept["category"].notna()].copy()
unlabeled = df_kept[df_kept["category"].isna()]

print(f"Labeled:   {len(labeled):,}")
print(f"Unlabeled: {len(unlabeled):,}")
print(f"\nClass distribution:")
print(labeled["category"].value_counts())

Rows after product filter: 58,635
Labeled:   27,695
Unlabeled: 30,940

Class distribution:
category
payment_issue      10007
billing_dispute     6799
account_issue       6641
fraud               4248
Name: count, dtype: int64


In [8]:
import re

def clean_text(text):
    """Basic text cleaning for complaint narratives."""
    if not isinstance(text, str):
        return ""
    
    # Replace CFPB redaction placeholders
    text = re.sub(r"X{2,}", " ", text)           # XXXX, XX, X
    text = re.sub(r"x{2,}", " ", text)           # lowercase variants
    
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text)
    
    # Strip leading/trailing
    text = text.strip()
    
    return text


# Apply cleaning to labeled rows only
labeled["clean_text"] = labeled["Consumer complaint narrative"].apply(clean_text)

# Check the length distribution
labeled["text_len"] = labeled["clean_text"].str.len()

print("Text length stats:")
print(labeled["text_len"].describe())

print(f"\nRows with < 20 chars (likely junk): {(labeled['text_len'] < 20).sum()}")
print(f"Rows with > 5000 chars (very long): {(labeled['text_len'] > 5000).sum()}")

Text length stats:
count    27695.000000
mean      1145.976313
std       1206.570552
min         13.000000
25%        521.000000
50%        770.000000
75%       1406.000000
max      31585.000000
Name: text_len, dtype: float64

Rows with < 20 chars (likely junk): 3
Rows with > 5000 chars (very long): 354


In [9]:
# Drop junk rows (too short)
labeled = labeled[labeled["text_len"] >= 20].copy()

# Re-check the distribution
print(f"Final rows: {len(labeled):,}\n")
print(f"Final class distribution:")
print(labeled["category"].value_counts())
print(f"\nClass balance (max/min ratio): {labeled['category'].value_counts().max() / labeled['category'].value_counts().min():.2f}x")

# Show 2 examples per class so we can sanity-check the cleaning
print("\n" + "="*80)
print("Sample examples per class:")
print("="*80)

for cat in labeled["category"].unique():
    examples = labeled[labeled["category"] == cat].head(2)
    print(f"\n### {cat.upper()} ###")
    for _, row in examples.iterrows():
        text = row["clean_text"][:200] + "..." if len(row["clean_text"]) > 200 else row["clean_text"]
        print(f"  {text}")

Final rows: 27,692

Final class distribution:
category
payment_issue      10007
billing_dispute     6797
account_issue       6641
fraud               4247
Name: count, dtype: int64

Class balance (max/min ratio): 2.36x

Sample examples per class:

### FRAUD ###
  Check x system is not the proper value to protect banking system. Someone illegally access my account. This person have my social and generated match sane id as drive lic to send me with social. This ...
  During a business trip to , / /2022 I attempted to make a cash withdrawal at an ATM on one of the major streets in tourist area. I selected an international ATM ( one out of four ATMs ) and inserted m...

### BILLING_DISPUTE ###
  American express has been ignoring my evidence regarding a dispute or 4 times now. I have even called in and escalated it, they told me over the phone that they see that the evidence has been ignored ...
  On / /year>, a payment of {$2100.00} was processed through my PayPal account for a purchase f

In [10]:
import re

# High-precision fraud overrides
FRAUD_PATTERNS = [
    r"\bunauthorized\b",
    r"\bunauthorised\b",
    r"\bdidn'?t authorize\b",
    r"\bdid not authorize\b",
    r"\bwithout my (permission|consent|authorization)\b",
    r"\bidentity theft\b",
    r"\bstolen\b",
    r"\bfraudulent\b",
    r"\bnever opened\b",
    r"\bfraud\b",
]

FRAUD_REGEX = re.compile("|".join(FRAUD_PATTERNS), re.IGNORECASE)

def override_label(row):
    """Relabel to 'fraud' if text has strong fraud signals."""
    text = row["clean_text"]
    if FRAUD_REGEX.search(text):
        return "fraud"
    return row["category"]


# Apply overrides
labeled["category_original"] = labeled["category"]
labeled["category"] = labeled.apply(override_label, axis=1)

# How many changed?
changed = (labeled["category"] != labeled["category_original"]).sum()
print(f"Rows relabeled to fraud: {changed:,} ({changed/len(labeled)*100:.1f}%)")

# New distribution
print(f"\nNew class distribution:")
print(labeled["category"].value_counts())

# Confusion: how many moved from each class
print(f"\nMoved from:")
moved = labeled[labeled["category"] != labeled["category_original"]]
print(moved["category_original"].value_counts())

Rows relabeled to fraud: 8,292 (29.9%)

New class distribution:
category
fraud              12539
payment_issue       5976
account_issue       4763
billing_dispute     4414
Name: count, dtype: int64

Moved from:
category_original
payment_issue      4031
billing_dispute    2383
account_issue      1878
Name: count, dtype: int64


In [11]:
# Revert: drop the overridden labels, keep the originals
labeled["category"] = labeled["category_original"]
labeled = labeled.drop(columns=["category_original"])

# Verify we're back to the original clean state
print(f"Rows: {len(labeled):,}")
print(f"\nClass distribution:")
print(labeled["category"].value_counts())
print(f"\nBalance ratio: {labeled['category'].value_counts().max() / labeled['category'].value_counts().min():.2f}x")

Rows: 27,692

Class distribution:
category
payment_issue      10007
billing_dispute     6797
account_issue       6641
fraud               4247
Name: count, dtype: int64

Balance ratio: 2.36x


In [13]:
print("Columns:", labeled.columns.tolist())
print(f"Rows: {len(labeled)}")
print(f"\nCategory distribution:")
print(labeled["category"].value_counts())

Columns: ['Product', 'Sub-product', 'Issue', 'Consumer complaint narrative', 'Complaint ID', 'category', 'clean_text', 'text_len']
Rows: 27692

Category distribution:
category
payment_issue      10007
billing_dispute     6797
account_issue       6641
fraud               4247
Name: count, dtype: int64


In [14]:
final = labeled[["Complaint ID", "clean_text", "category"]].copy()

print(f"Final dataset: {len(final):,} rows")
print(f"\nClasses:")
print(final["category"].value_counts())

final.to_csv("../data/processed/complaints_clean.csv", index=False)
print(f"\nSaved to data/processed/complaints_clean.csv")

Final dataset: 27,692 rows

Classes:
category
payment_issue      10007
billing_dispute     6797
account_issue       6641
fraud               4247
Name: count, dtype: int64

Saved to data/processed/complaints_clean.csv


In [15]:
import pandas as pd
check = pd.read_csv("../data/processed/complaints_clean.csv")
print(f"Rows: {len(check):,}")
print(f"Columns: {check.columns.tolist()}")
print(f"\nClasses:")
print(check["category"].value_counts())
print(f"\nFirst 2 rows:")
check.head(2)

Rows: 27,692
Columns: ['Complaint ID', 'clean_text', 'category']

Classes:
category
payment_issue      10007
billing_dispute     6797
account_issue       6641
fraud               4247
Name: count, dtype: int64

First 2 rows:


,Complaint ID,clean_text,category
0,20367980,Check x system is not the proper value to protect banking system. Someone illegally access my account. This person have my social and generated ma...,fraud
1,7314325,"During a business trip to , / /2022 I attempted to make a cash withdrawal at an ATM on one of the major streets in tourist area. I selected an int...",fraud
